In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd

In [ ]:
fichier_csv = "/content/drive/MyDrive/LOC_2.csv"

In [ ]:
df = pd.read_csv(fichier_csv, sep=";", encoding="utf-8-sig")

In [ ]:
df

,token_val,latitude,longitude,addresstype
0,Espagne,40.4637,-3.7492,country
1,France,46.2276,2.2137,country
2,Royaume-Uni,55.3781,-3.4360,country
3,Bam,29.1061,58.3681,city
4,Djam,34.3981,64.5153,archaeological_site
...,...,...,...,...
978,îles Marquises,-9.0000,-139.5000,archipelago
979,îles Marshall,7.1315,171.1845,country
980,îles Salomon,-9.6457,160.1562,country
981,îles Tsonga,NaN,NaN,NaN


In [ ]:
from geopy.geocoders import Nominatim

In [ ]:
geolocator = Nominatim(user_agent="cours_python", timeout=200) # On augmente le temps

In [ ]:
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=4, error_wait_seconds=4)

In [ ]:
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable, GeocoderRateLimited
# On importe la bibliothèque time
import time

In [ ]:
def geocodeur(toponyme, max_retries=3):
    # On affiche le nom du toponyme choisi
    print(f" > géocodage {toponyme}")
    # Pour chaque tentative dans le nombre maximum d'essai
    for attempt in range(max_retries):
        # On essaie ce bloc
        try:
            # Utiliser l'objet 'geocode' qui est configuré avec RateLimiter
            loc = geocode(toponyme)
            # Si le toponyme existe
            if loc is not None:
                # On affiche la latitude et la longitude du toponyme en question
                print(f" lat : {loc.latitude}; long : {loc.longitude}")
                print(loc.raw)
                # Le résultat de cette fonction sont les coordonnées du toponyme en input
                return loc.latitude, loc.longitude, loc.raw['addresstype']
        # Gestion des exceptions : si le géocodeur met trop de temps
        # ou si le service de géocodage n'est pas disponible, ou si la limite de taux est atteinte
        except (GeocoderTimedOut, GeocoderUnavailable, GeocoderRateLimited) as e:
            print(f"La tentative {attempt+1} a échoué ({e}), on réessaie dans 5 sec.")
            # On ajoute un temps de pause de 5 secondes
            time.sleep(5)
    print(f"Échec du géocodage pour {toponyme} après {max_retries} tentatives.")
    return None, None, None # Retourne None si toutes les tentatives échouent

In [ ]:
df_sans_doublons = df[df["token_val"].notna()].drop_duplicates(subset=["token_val"], keep="first").copy()

In [ ]:
successful_geocoding_results = []
failed_geocoding_results = []

In [ ]:
for index, row in df_sans_doublons.iterrows():
    token = row['token_val']
    lat, lon, addresstype = geocodeur(token)

    dico = {
        'token': token,
        'latitude': lat,
        'longitude': lon,
        'addresstype': addresstype
    }

    if lat is None:
        failed_geocoding_results.append(dico)
    else:
        successful_geocoding_results.append(dico)

 > géocodage Espagne
 lat : 39.3260685; long : -4.8379791
{'place_id': 297465031, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'relation', 'osm_id': 1311341, 'lat': '39.3260685', 'lon': '-4.8379791', 'class': 'boundary', 'type': 'administrative', 'place_rank': 4, 'importance': 0.9295913704622313, 'addresstype': 'country', 'name': 'España', 'display_name': 'España', 'boundingbox': ['27.4335426', '43.9933088', '-18.3936845', '4.5918885']}
 > géocodage France
 lat : 46.603354; long : 1.8883335
{'place_id': 300091621, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'relation', 'osm_id': 2202162, 'lat': '46.6033540', 'lon': '1.8883335', 'class': 'boundary', 'type': 'administrative', 'place_rank': 4, 'importance': 0.9781609870055644, 'addresstype': 'country', 'name': 'France', 'display_name': 'France', 'boundingbox': ['-50.2187169', '51.3055721', '-178.3873749', '172.3057152']}
 > géocodage Roy

KeyboardInterrupt: 

In [ ]:
df_res= pd.DataFrame(res)

In [ ]:
df_res

,token,latitude,longitude,addresstype
0,Vallée de Bamiyan,None,None,None
1,Abou Tamsa,None,None,None
2,Abou Tbeirah,None,None,None
3,Abousir,None,None,None
4,Abu Tubairah,None,None,None
...,...,...,...,...
132,ville royale de Megiddo,None,None,None
133,volcan du Teide,None,None,None
134,Églises de Moldavie,None,None,None
135,île de Faylakha,None,None,None


In [ ]:
df_res.to_csv('loc_triees_geocodees.csv', index=False)

In [ ]:
from google.colab import files

files.download('loc_triees_geocodees.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_res_partial = pd.DataFrame(res)
display(df_res_partial.head())
print(f"Created a DataFrame with {len(df_res_partial)} partial results.")

df_res_partial.to_csv('loc_triees_geocodees_partial.csv', index=False)
print("Partial results saved to 'loc_triees_geocodees_partial.csv'.")

""


Created a DataFrame with 0 partial results.
Partial results saved to 'loc_triees_geocodees_partial.csv'.
